In [2]:
import polars as pl
from dotenv import load_dotenv
import os

load_dotenv(".env")

from src.repository.alarm_graph_repository import AlarmGraphRepository

lazy_frame = pl.scan_parquet("data/raw/alarm_history_dump.parquet")
graph_repo = AlarmGraphRepository(os.getenv("HISTORY_DB_PATH"))

In [ ]:
from src.pipelines.simple_time_correlation import SimpleTimeCorrelationHistory

SimpleTimeCorrelationHistory.train(lazy_frame, graph_repo, threshold_minutes=5)

Processando nós:   0%|          | 4/912 [04:57<29:18:51, 116.22s/nó]

In [ ]:
from src.postprocess.enumerate_incidents import EnumerateIncidents

EnumerateIncidents.enumerate_data(graph_repo)

graph_repo.preview_nodes()

Calculando WCC: 100%|█████████▉| 908/912 [01:35<00:00, 26.58nó/s]

In [4]:
from src.utils.node_summary import node_summary
from src.repository.aggregate_results_repository import AggregateResultsRepository

summary, general_metrics = node_summary(graph_repo)

results_repo = AggregateResultsRepository(filename="simple_time_corr_history")
results_repo.save(summary)
results_repo.load()

✅ Nova versão salva com sucesso em: data/results/simple_time_corr_history_20260530_140019.csv
📖 Carregando a versão mais recente encontrada: simple_time_corr_history_20260530_140019.csv


Node ID,Total de Alarmes,Total de Correlações,Total de Incidentes,Média de Alarmes por Incidente,Densidade
str,i64,i64,i64,f64,f64
"""a5126d4f-0bd1-4023-ae9d-d2c1e9…",232712,8584164,4090,56.8978,0.000317
"""7510d37a-4c5f-40da-b987-3977c2…",32148,119403,4073,7.892954,0.000231
"""8bf0d4d2-1cd3-4341-9b79-794833…",31766,116570,4065,7.814514,0.000231
"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…",32023,119030,4054,7.899112,0.000232
"""75577cb5-6565-42da-a042-d9603e…",27157,83723,4041,6.720366,0.000227
…,…,…,…,…,…
"""e9488358-73d3-4612-a413-62ac2b…",714,0,1,714.0,0.0
"""0b466dd2-3057-4816-9379-ac1e7e…",12,0,1,12.0,0.0
"""0ea468f1-9c1d-4386-9ea8-8ba975…",711,0,1,711.0,0.0


In [5]:
graph_repo.close()